# Simulation Code T = 2

## Import libraries

In [1]:

from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszBradic import estimateBradic
import torch
import pandas as pd
import time

Unable to determine R home: [Errno 2] No such file or directory: 'R'
Error importing in API mode: ImportError('libR.so: cannot open shared object file: No such file or directory')
Trying to import in ABI mode.


ValueError: r_home is None. Try python -m rpy2.situation

In [2]:
python -m rpy2.situation

SyntaxError: invalid syntax (3377621633.py, line 1)

## Estimation Settings

In [ ]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : 0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

rf_f_settings = {
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

#### Simulation settings:

In [ ]:
torch.manual_seed(123);

# Parameters
N = 200
tmax = 1

# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 3
dimX1 = 5
dimX2 = 5

#### Define treatment probability function

In [ ]:
# Bounds (only for truncated distributions)
lower = 0.10
upper = 0.90

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Define func_link function (nonlinear probability function from Bradic)
def func_link(x):
    return (x > 0) * torch.abs(x) / (torch.abs(x) + 1) + (x < 0) / (torch.abs(x) + 1)

# Define a truncated func_link function
def truncated_link(x):
    return lower + (upper - lower) * func_link(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)

# Simple nonlinear probability (from adversarial Riesz paper)
def double_nonlinear(x):
    return lower + (upper - lower) * ((x < - 1/2) + (x < 1/2))

In [ ]:
treatment_probability_func = logistic

#### Generate data

In [ ]:
# Coefficients
beta_pi1_0 = 0
beta_pi1_S1 = torch.tensor([1, 1, 1] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # gamma_a i(paper)
beta_pi2_0_0 = 0
beta_pi2_0_S1 = torch.tensor([0.5, 0, -0.5] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a' first part (paper)
beta_pi2_0_S2 = torch.tensor([0.5, 0, 0.5] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a' second part (paper)
beta_pi2_1_0 = 0
beta_pi2_1_S1 = torch.tensor([1, 1, 0] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # eta_a first part (paper)
beta_pi2_1_S2 = torch.tensor([1, -1, 0] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # eta_a second part (paper)
beta_g0_0 = 1
beta_g0_S1 = torch.tensor([1, 1, -1] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a' first part (paper)
beta_g0_S2 = torch.tensor([1, 1, 1] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a' second part (paper)
beta_g1_0 = -1
beta_g1_S1 = torch.tensor([-1, 1, -1] + [0] * (dimX1 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a first part (paper)
beta_g1_S2 = torch.tensor([-1, -1, 1] + [0] * (dimX2 - 3), dtype=torch.float32).reshape(-1,1) # alpha_a second part (paper)

# Period 1
S1_all = torch.randn(N * tmax, dimX1)
pi1_all = treatment_probability_func(beta_pi1_0 + S1_all @ beta_pi1_S1).reshape(-1,1)
A1_all = torch.bernoulli(pi1_all).int().reshape(-1,1)

# Period 2
delta1_all = torch.randn(N * tmax,1)
delta2_all = torch.randn(N * tmax, dimX2)

S2_all = S1_all + A1_all * (1 + delta1_all) + delta2_all
S2_1_all = S1_all + 1 + delta1_all + delta2_all
S2_0_all = S1_all + delta2_all

pi2_0_all = treatment_probability_func(beta_pi2_0_0 + S1_all @ beta_pi2_0_S1 + S2_0_all @ beta_pi2_0_S2)
pi2_1_all = treatment_probability_func(beta_pi2_1_0 + S1_all @ beta_pi2_1_S1 + S2_1_all @ beta_pi2_1_S2)
pi2_all = (1 - A1_all) * pi2_0_all + A1_all * pi2_1_all

A2_all = torch.bernoulli(pi2_all).int() 

# Outcome
g_all = ((A1_all + A2_all == 0).float() * (beta_g0_0 + S1_all @ beta_g0_S1 + S2_all @ beta_g0_S2) + 
         (A1_all * A2_all == 1).float() * (beta_g1_0 + S1_all @ beta_g1_S1 + S2_all @ beta_g1_S2))
zeta_all = torch.randn(N * tmax,1)
Y_all = g_all + zeta_all

#### True values:

In [ ]:
mu2_1_all = beta_g1_0 + S1_all @ beta_g1_S1 + S2_1_all @ beta_g1_S2
mu2_0_all = beta_g0_0 + S1_all @ beta_g0_S1 + S2_0_all @ beta_g0_S2
mu1_1_all = beta_g1_0 + S1_all @ (beta_g1_S1 + beta_g1_S2) + beta_g1_S2.sum()
mu1_0_all = beta_g0_0 + S1_all @ (beta_g0_S1 + beta_g0_S2)

theta0 = beta_g0_0
theta1 = beta_g1_0 + beta_g1_S2.sum()
theta = theta1 - theta0
print(theta, theta1, theta0)


tensor(-3.) tensor(-2.) 1


In [ ]:
torch.mean(mu1_0_all)

tensor(1.1679)

## Estimation:

#### Estimation settings

In [ ]:
folds = 4

time0 = time.time()

#### Estimation 

In [ ]:
pred_theta = torch.zeros(tmax, 11)
pred_sig = torch.zeros(tmax, 11)

RR1 = torch.zeros(N, tmax, 11)
RR2 = torch.zeros(N, tmax, 11)
f1 = torch.zeros(N, tmax, 11)
f2 = torch.zeros(N, tmax, 11)

for t in range(0,tmax):

    # Get data for current iteration
    S1, S2 = S1_all[(t) * N : (t+1) * N, :], S2_all[(t) * N : (t+1) * N, :]
    A1, A2 = A1_all[(t) * N : (t+1) * N], A2_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]

    X = torch.hstack((S1,S2))
    X_index = torch.tensor([S1.shape[1]-1,S1.shape[1]+S2.shape[1]-1])
    D = torch.hstack((A1,A2))

    # Set counterfactual of interest
    d = 1 * torch.ones(D.shape) 

    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    pi1, pi2_0, pi2_1  = pi1_all[(t) * N : (t+1) * N], pi2_0_all[(t) * N : (t+1) * N], pi2_1_all[(t) * N : (t+1) * N]
    mu2_1, mu2_0, mu1_1, mu1_0 = mu2_1_all[(t) * N : (t+1) * N], mu2_0_all[(t) * N : (t+1) * N], mu1_1_all[(t) * N : (t+1) * N], mu1_0_all[(t) * N : (t+1) * N]
    if d[0,0] == 1:
        RR2[:,t,:1], RR1[:,t,:1] = A1 * A2 / (pi1 * pi2_1), A1 / pi1 
        theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_1   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_1)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
    elif d[0,0] == 0:
        RR2[:,t,:1], RR1[:,t,:1] = (1 - A1) * (1 - A2) / ((1 - pi1  ) * (1 - pi2_0  )), (1 - A1) / (1 - pi1)
        theta_hat = (RR2[:,t,:1] * Y - (RR1[:,t,:1] - 1) * mu1_0   - (RR2[:,t,:1] - RR1[:,t,:1]) * mu2_0)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))

    #---------------------------------------------------------------------------
    # Estimator 2: Bradic
    # bradic_result = estimateBradic(Y, X, D, X_index, folds)
    # if d[0,0] == 1:
    #     pred_theta[t,1], pred_sig[t,1] = bradic_result[7], torch.sqrt(torch.tensor(bradic_result[10]))
    # elif d[0,0] == 0:
    #     pred_theta[t,1], pred_sig[t,1] = bradic_result[13], torch.sqrt(torch.tensor(bradic_result[16]))

    #---------------------------------------------------------------------------
    # Estimator 3: LASSO, RF, Net

    pred_theta[t,2:], pred_sig[t,2:], RR1[:,t,2:], RR2[:,t,2:] , f1[:,t,2:], f2[:,t,2:] = estimateDynamicRiesz_all(Y, X, D, d, X_index, folds)

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


Time until iteration  0 :  687.4199912548065
Finished. Total time:  687.4203324317932


In [ ]:
%debug

> /Users/wisserutgers/Documents/Uni/PhD/Harvard/RA Rahul Singh/Code_v2/utils/dynamicRieszFunctions.py(346)estimateDynamicRiesz_all()
    344                                                     net_f_settings = net_f_settings_global):
    345 
--> 346     fold_results = torch.zeros(Y.shape,3)
    347     RR1 = torch.zeros(Y.shape,3)
    348     RR2 = torch.zeros(Y.shape,3)

torch.Size([500, 1])
torch.Size([500, 1])
--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user


## Compute statistics

#### Get true value

In [ ]:
true_value = theta1

#### Compute statistics

In [ ]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", 
               "LASSO-LASSO", "RF-LASSO", "Net-LASSO",
               "LASSO-RF", "RF-RF", "Net-RF",
               "LASSO-Net", "RF-Net", "Net-Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

#### Print results

In [ ]:
df = pd.DataFrame(data)
print(df)

         Method      Bias      RMSE  Coverage  Interval Length
0        Oracle -0.337980  0.337980       1.0         1.002398
1        Bradic  2.000000  2.000000       0.0         0.000000
2   LASSO-LASSO -0.163115  0.163115       1.0         1.272289
3      RF-LASSO -0.105576  0.105576       1.0         1.081290
4     Net-LASSO -0.286078  0.286078       1.0         1.218498
5      LASSO-RF -0.126708  0.126708       1.0         1.791882
6         RF-RF -0.415637  0.415637       1.0         1.222788
7        Net-RF -0.122507  0.122507       1.0         1.601546
8     LASSO-Net -0.160528  0.160528       1.0         1.398371
9        RF-Net -0.085364  0.085364       1.0         1.089524
10      Net-Net -0.231296  0.231296       1.0         1.404162


In [ ]:
pred_theta

tensor([[-2.3380,  0.0000, -2.1631, -2.1056, -2.2861, -2.1267, -2.4156, -2.1225,
         -2.1605, -2.0854, -2.2313]])

In [ ]:
# check_t = 0
# Compare RR:
# pd.DataFrame(torch.hstack((RR1[:,check_t,0:1], RR1[:,check_t,2:3], RR1[:,check_t,3:4], RR1[:,check_t,4:5]))).head(50)
# pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)
